*** 데이터 다운받을 때 YOLO Segmentation 1.1** 로 다운받으셈 아 1시간동안 헛발질 ㅡㅡ

In [ ]:
# YOLOv8-seg (bed polygon) training starter
# - 목적: 180장 라벨링된 베드(segmentation)로 train/val/test split → 학습 → 결과 확인(예시 이미지 저장)
# - 가정: 원본 데이터가 YOLO 포맷으로 정리되어 있음
#   dataset_root/
#     images/   (jpg/png)
#     labels/   (txt)  # YOLO-seg polygon 포맷: cls x1 y1 x2 y2 ... (모두 0~1 정규화)
# - 주의: 현재 라벨 예시에서 class id가 18로 되어 있음. 이 스크립트는 'bed 단일 클래스' 모델을 위해
#         라벨 class를 0으로 remap하여 복사함.

from __future__ import annotations

import os
import random
import shutil
from pathlib import Path
from typing import List, Tuple

# =========================
# 0) 사용자 설정
# =========================
# TODO: 네 환경에 맞게 바꿔
DATASET_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX")  # 여기에 images/, labels/가 있어야 함
OUT_ROOT     = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/results")     # split 결과가 생성될 폴더

SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

# bed 단일 클래스 학습용 라벨 remap
SOURCE_CLASS_ID = 18
TARGET_CLASS_ID = 0


In [ ]:

# =========================
# 1) train/val/test split
# =========================

def list_image_stems(images_dir: Path, exts=(".jpg", ".jpeg", ".png", ".bmp", ".webp")) -> List[str]:
    stems = []
    for p in images_dir.iterdir():
        if p.is_file() and p.suffix.lower() in exts:
            stems.append(p.stem)
    stems.sort()
    return stems


def ensure_dirs(base: Path):
    for split in ["train", "val", "test"]:
        (base / "images" / split).mkdir(parents=True, exist_ok=True)
        (base / "labels" / split).mkdir(parents=True, exist_ok=True)


def remap_label_file(src_txt: Path, dst_txt: Path, src_id: int = 18, dst_id: int = 0) -> None:
    """YOLO-seg 라벨의 클래스 id만 바꿔서 저장.
    - 한 파일에 여러 객체가 있을 수 있으므로 각 라인 첫 토큰만 확인.
    """
    if not src_txt.exists():
        # 라벨 없으면 빈 파일 생성 (학습 시 background로 취급)
        dst_txt.write_text("", encoding="utf-8")
        return

    out_lines = []
    for line in src_txt.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        # 첫 토큰이 class id
        try:
            cls = int(float(parts[0]))
        except Exception:
            # 이상한 라인은 그대로 통과(안전)
            out_lines.append(line)
            continue

        if cls == src_id:
            parts[0] = str(dst_id)
        # 만약 다른 class가 섞여있다면 그대로 남김(혹은 여기서 제거하고 싶으면 continue 처리)
        out_lines.append(" ".join(parts))

    dst_txt.write_text("\n".join(out_lines) + ("\n" if out_lines else ""), encoding="utf-8")


def split_and_copy(dataset_root: Path, out_root: Path) -> Tuple[List[str], List[str], List[str]]:
    images_dir = dataset_root / "images"
    labels_dir = dataset_root / "labels"

    assert images_dir.exists(), f"images 폴더가 없음: {images_dir}"
    assert labels_dir.exists(), f"labels 폴더가 없음: {labels_dir}"

    stems = list_image_stems(images_dir)
    assert len(stems) > 0, f"이미지가 없음: {images_dir}"

    # 라벨 없는 이미지가 섞여있으면 경고 (학습은 가능하지만 의도 확인 필요)
    missing_labels = [s for s in stems if not (labels_dir / f"{s}.txt").exists()]
    if missing_labels:
        print(f"[WARN] 라벨 없는 이미지가 {len(missing_labels)}장 있음. 예: {missing_labels[:5]}")

    random.seed(SEED)
    random.shuffle(stems)

    n = len(stems)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    # 나머지를 test로
    n_test  = n - n_train - n_val

    train_stems = stems[:n_train]
    val_stems   = stems[n_train:n_train + n_val]
    test_stems  = stems[n_train + n_val:]

    print(f"총 {n}장 → train {len(train_stems)}, val {len(val_stems)}, test {len(test_stems)}")

    # 출력 폴더 초기화(원하면 주석 처리)
    if out_root.exists():
        shutil.rmtree(out_root)
    ensure_dirs(out_root)

    def copy_one(stem: str, split: str):
        # 이미지 복사
        src_img = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            cand = images_dir / f"{stem}{ext}"
            if cand.exists():
                src_img = cand
                break
        if src_img is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없음: {stem}")

        dst_img = out_root / "images" / split / src_img.name
        shutil.copy2(src_img, dst_img)

        # 라벨 remap 복사
        src_txt = labels_dir / f"{stem}.txt"
        dst_txt = out_root / "labels" / split / f"{stem}.txt"
        remap_label_file(src_txt, dst_txt, src_id=SOURCE_CLASS_ID, dst_id=TARGET_CLASS_ID)

    for s in train_stems:
        copy_one(s, "train")
    for s in val_stems:
        copy_one(s, "val")
    for s in test_stems:
        copy_one(s, "test")

    return train_stems, val_stems, test_stems


train_stems, val_stems, test_stems = split_and_copy(DATASET_ROOT, OUT_ROOT)


[WARN] 라벨 없는 이미지가 20장 있음. 예: ['bed21_20251221_083915_cam2', 'bed22_20251204_075400_cam2', 'bed22_20251209_080049_cam2', 'bed22_20251214_075453_cam2', 'bed22_20251219_052249_cam2']
총 380장 → train 304, val 38, test 38


In [ ]:

# =========================
# 1-1) data.yaml 생성
# =========================

def write_data_yaml(out_root: Path, yaml_path: Path):
    # Ultralytics는 절대경로/상대경로 모두 가능
    yaml_text = f"""# YOLOv8-seg dataset config
path: {out_root.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: bed
"""
    yaml_path.write_text(yaml_text, encoding="utf-8")
    print(f"data.yaml 저장: {yaml_path}")

DATA_YAML = OUT_ROOT / "data.yaml"
write_data_yaml(OUT_ROOT, DATA_YAML)


data.yaml 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/results/data.yaml


### yolov8 seg 학습

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import SegmentationModel # Import the class mentioned in the error
from ultralytics.nn.modules import Conv # Import Conv as well

# Add the following line to address the UnpicklingError
torch.serialization.add_safe_globals([SegmentationModel, torch.nn.modules.container.Sequential, Conv])

# 모델 선택: 가볍게 시작하려면 yolov8n-seg, 좀 더면 yolov8s-seg
MODEL_NAME = "yolov8n-seg.pt"

# 학습 하이퍼파라미터 (오늘 목표: 빠르게 성능 확인)
EPOCHS = 80
IMGSZ  = 960   # 베드가 큰 구조라 해상도 너무 낮추면 polygon이 무너질 수 있음
BATCH  = 8     # GPU 메모리에 맞도 조절
DEVICE = 0     # GPU:0, CPU는 'cpu'

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    seed=SEED,
    overlap_mask=False,
    project=str(OUT_ROOT / "runs"),
    name="bed_seg_v1",
    pretrained=True,
    patience=20,     # 조기종료
    verbose=True,
)

# =========================
# 2-1) 성능 요약 출력
# =========================
# Ultralytics는 학습 끝나면 runs/segment/.. 아래 results.csv, confusion_matrix 등 생성

RUN_DIR = Path(results.save_dir)  # e.g., /content/bed_yolo/runs/segment/bed_seg_v1
print("\n[RUN_DIR]", RUN_DIR)

# 주요 결과 파일 위치 안내
print("- results.png:", RUN_DIR / "results.png")
print("- confusion_matrix.png:", RUN_DIR / "confusion_matrix.png")
print("- val batch 예측 예시:", RUN_DIR / "val_batch0_pred.jpg")


Ultralytics 8.4.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/results/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bed_seg_v1, nbs=64, n

In [ ]:

# =========================
# 2-2) test에서 예시 이미지 예측 & 저장
# =========================
# 원하는 만큼 뽑아서 예측 결과 이미지를 저장해 확인

import glob

best_pt = RUN_DIR / "weights" / "best.pt"
assert best_pt.exists(), f"best.pt 없음: {best_pt}"

infer_model = YOLO(str(best_pt))

# test 이미지 중 랜덤 8장
test_images = sorted(glob.glob(str(OUT_ROOT / "images" / "test" / "*")))
random.seed(SEED)
sample_imgs = random.sample(test_images, k=min(8, len(test_images)))

PRED_DIR = OUT_ROOT / "pred_samples"
PRED_DIR.mkdir(parents=True, exist_ok=True)

pred_results = infer_model.predict(
    source=sample_imgs,
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.5,
    device=DEVICE,
    save=True,
    project=str(PRED_DIR),
    name="bed_seg_v1_test_samples",
)

print("\n[예시 예측 저장 폴더]", PRED_DIR / "bed_seg_v1_test_samples")

# (Colab) 첫 예시 이미지 화면 출력
# 로컬 주피터면 아래 블록으로 확인 가능
try:
    from PIL import Image
    from IPython.display import display

    pred_img_paths = sorted((PRED_DIR / "bed_seg_v1_test_samples").glob("*.jpg"))
    if pred_img_paths:
        display(Image.open(pred_img_paths[0]))
except Exception as e:
    print("이미지 출력 생략:", e)

# =========================
# (선택) 3) 판단 기준 가이드
# =========================
# 오늘 결론을 내릴 때는 아래만 보면 됨:
# - val mAP50-95(세그) 수치가 상승 추세인지
# - val_batch*_pred.jpg에서 베드 사다리꼴이 안정적으로 잡히는지
# - 경계가 흔들리면 (1) 라벨 추가, (2) imgsz↑, (3) 모델 크기(s-seg)↑ 를 우선 고려


Output hidden; open in https://colab.research.google.com to view.